Il problema è di classificare ogni paziente in due classi:
0 = diabete non presente
1 = diabete presente

Il vostro primo obiettivo è quello di arrivare a compiere l’operazione di classificazione nel modo più rapido possibile: 
non concentratevi sull’accuratezza o sul fare tutti gli step possibili nella fase di pulizia dei dati ma sull’avere un codice “minimo” che almeno funzioni. In base al tempo a vostra disposizione sceglierete eventualmente altri aspetti su cui concentrarvi.

La descrizione delle variabili del dataset è disponibile presso: https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset?select=diabetes_prediction_dataset.csv


In [ ]:
# Librerie e percorso alla lettura del dataset
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

root = Path.cwd()
file_path = root / 'diabetes_prediction_dataset.csv'

In [ ]:
# --- importo il dataset
df = pd.read_csv(file_path)

# visualizzo informazioni generali
df.info()

I dati includono caratteristiche quali 
sesso, età, ipertensione, malattie cardiache, storia di fumo, indice di massa corporea (BMI), livello di HbA1c, livello di glucosio nel sangue e diabete.

In [ ]:
# visualizzo le prime 10 righe del dataset
df.head(10)

In [ ]:
# visualizzo le statistiche
df.describe()

In [ ]:
# visualizzo se ci sono valori nulli
df.isnull().sum()

In [ ]:
# visualizzo lunghezza del dataset
df.shape

In [ ]:
# controllo se ci sono duplicati
is_duplicate = df.duplicated().sum()
print(f"Righe duplicate esatte: {is_duplicate}")
print(f"Percentuale: {is_duplicate / len(df) * 100:.2f}%")

In [ ]:
# procedo all'eliminazione dei duplicati
print(f"Righe originali:  {len(df)}")
df = df.drop_duplicates()
print(f"Righe dopo pulizia: {len(df)}")
print(f"Duplicati rimossi:  {100000 - len(df)}")

In [ ]:
# prima dell'encoding analizzo la colonna smoking_history
print(df["smoking_history"].unique())       # mostra i valori distinti
print(df["smoking_history"].value_counts(normalize=True) *100) # mi da il dettaglio anche di quante righe ha ciascuno valore

In [ ]:
# prima dell'encoding analizzo la colonna gender
print(df["gender"].unique())       # mostra i valori distinti
print(df["gender"].value_counts(normalize=True)*100) # mi da il dettaglio anche di quante righe ha ciascuno valore

**Considerazioni:**

- Per la colonna `smoking_history` ho rilevato che si sono 32887 `No Info ` e ha un peso più del 34%
  - mi viene difficile pensare di droppare o assegnare un valore con l'uso della media o della moda.
  - credo di codificarlo così com'è e sarà il modello a gestire il pattern di correlazione con il diabete.
- Per la colonna `gender` ha un peso insignificante dello 0.018722
  - escluto da gender tutto quello che è != Other

In [ ]:
# Nessuna sostituzione di "No Info", encoding diretto di tutto
le_smoking = LabelEncoder()
df["smoking_history"] = le_smoking.fit_transform(df["smoking_history"])

# escludo Other
df = df[df["gender"] != "Other"]

# Encoding gender
le_gender = LabelEncoder()
df["gender"] = le_gender.fit_transform(df["gender"])

# Verifica
print(df["smoking_history"].value_counts().sort_values(ascending=False))
print(df["gender"].value_counts())

In [ ]:
# --- SPLIT TRAIN/TEST

X = df.drop(columns=["diabetes"])  # tutto tranne il diabetes → features
y = df["diabetes"]                 # solo il diabetes → target

print(X.shape)  # (96128, 8)
print(y.shape)  # (96128, )

print(f"\nDistribuzione classi:\n{y.value_counts(normalize=True).round(3)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y # mantiene le stesse proporzioni (91,2% vs 8,8%)
)

In [ ]:
# --- MODELLO
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"   # gestisce lo sbilanciamento delle classi
)
rf.fit(X_train, y_train)

In [ ]:
# --- VALUTAZIONE
y_pred = rf.predict(X_test)             # Chiede al modello di dare una risposta netta 0 | 1
y_prob = rf.predict_proba(X_test)[:, 1] # Chiede invece la probabilità

print("--- Report classificazione ---")
print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")

# roc_auc_score:
# 0.50 → modello inutile, equivale a tirare una moneta
# 0.85 → buono
# 0.95 → ottimo
# 1.00 → perfetto (quasi sempre sospetto, probabilmente overfitting)

In [ ]:
# Matrice di confusione
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No diabete", "Diabete"],
            yticklabels=["No diabete", "Diabete"])
plt.title("Matrice di confusione")
plt.ylabel("Reale")
plt.xlabel("Predetto")
plt.tight_layout()
plt.savefig("confusion_matrix.png")
plt.show()

In [ ]:
# --- PREDIZIONE SU NUOVO PAZIENTE
nuovo_paziente = pd.DataFrame([{
    "gender": le_gender.transform(["Male"])[0],
    "age": 46,
    "hypertension": 0,
    "heart_disease": 0,
    "smoking_history": le_smoking.transform(["former"])[0],
    "bmi": 30.5,
    "HbA1c_level": 6.8,
    "blood_glucose_level": 145
}])

prob = rf.predict_proba(nuovo_paziente)[0][1]
print(f"\nProbabilità diabete: {prob:.2%}")